# IADD Traffic Detection - YOLOv11 (Run 4)

Changes vs Run 3:
- Input resolution 960 -> 1280. Run 3 converged at 960; 1280 targets the small-object
  classes (person, traffic_light) that were the weakest.
- Batch 16 -> 8 so imgsz 1280 fits in the free T4's 15 GB.
- Trained from scratch (COCO-pretrained yolo11s), not resumed from Run 3.

The dataset, the leakage-free by-video split and the augmentation are unchanged, so
Run 4 vs Run 3 isolates the effect of input resolution.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 3. Mount Google Drive

The run folder lives on Drive so a limited free session can be continued on another
Google account: share the `iadd_runs` folder and the dataset zip, add a shortcut to
that account's My Drive root, then run every cell - training resumes from `last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Unzip the dataset into Colab

Colab's local disk is wiped on reconnect, so re-run this cell after any reconnect.

In [ ]:
!rm -rf /content/iadd_subset_v3
!unzip -q "/content/drive/MyDrive/iadd_subset_v3.zip" -d /content
!ls /content/iadd_subset_v3

## 5. Train - fresh, or auto-resume from Drive

If a checkpoint exists on Drive it resumes from the last epoch; otherwise it starts a
fresh Run 4. This is what makes multi-account training work: switch account, re-run,
and it continues where the previous session stopped.

In [ ]:
from ultralytics import YOLO
import os

MODEL    = 'yolo11s.pt'
IMGSZ    = 1280
BATCH    = 8          # fits imgsz 1280 on a 15 GB T4; raise to 12 if memory allows
EPOCHS   = 150
PATIENCE = 20

DRIVE_PROJECT = '/content/drive/MyDrive/iadd_runs'
RUN_NAME      = 'run4'
DATA          = '/content/iadd_subset_v3/data.yaml'
CKPT          = f'{DRIVE_PROJECT}/{RUN_NAME}/weights/last.pt'

if os.path.exists(CKPT):
    print('checkpoint found -> resuming:', CKPT)
    model = YOLO(CKPT)
    results = model.train(resume=True)
else:
    print('no checkpoint -> fresh run:', MODEL, '| imgsz', IMGSZ, '| batch', BATCH)
    model = YOLO(MODEL)
    results = model.train(
        data=DATA, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
        project=DRIVE_PROJECT, name=RUN_NAME, exist_ok=True, plots=True,
    )

## 6. Progress so far (optional)

Prints the per-epoch metrics table - useful after a reconnect to see the trend.

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/iadd_runs/run4/results.csv')
df.columns = [c.strip() for c in df.columns]
print('epochs done:', len(df))
df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
    'metrics/precision(B)', 'metrics/recall(B)']].tail(10)

## 7. Evaluate on validation

Standard Ultralytics validation of the best checkpoint: COCO-standard mAP over all
confidences, the per-class table, and the confusion / PR / F1 plots saved to Drive.

In [ ]:
from ultralytics import YOLO

DATA = '/content/iadd_subset_v3/data.yaml'
best = YOLO('/content/drive/MyDrive/iadd_runs/run4/weights/best.pt')

m = best.val(data=DATA, split='val', plots=True,
             project='/content/drive/MyDrive/iadd_runs', name='run4_val', exist_ok=True)
print('VAL  mAP50=%.4f  mAP50-95=%.4f  P=%.4f  R=%.4f'
      % (m.box.map50, m.box.map, m.box.mp, m.box.mr))

## 8. Evaluate on test

The test videos were never seen in training (leakage-free split), so this is the
number to report.

In [ ]:
mt = best.val(data=DATA, split='test', plots=True,
              project='/content/drive/MyDrive/iadd_runs', name='run4_test', exist_ok=True)
print('TEST mAP50=%.4f  mAP50-95=%.4f  P=%.4f  R=%.4f'
      % (mt.box.map50, mt.box.map, mt.box.mp, mt.box.mr))

## 9. Demo predictions - saved to run4/demo

Runs the model on a few test images and writes the annotated images to a folder on
Drive. Nothing is displayed inline, to keep the log clean.

In [ ]:
import glob

imgs = sorted(glob.glob('/content/iadd_subset_v3/images/test/*.jpg'))[:16]
best.predict(imgs, save=True,
             project='/content/drive/MyDrive/iadd_runs/run4', name='demo', exist_ok=True)
print('demo images saved -> iadd_runs/run4/demo')

---
Results on Drive under `iadd_runs/`:
- `run4/` - training, weights (`best.pt`, `last.pt`), and `demo/`
- `run4_val/`, `run4_test/` - evaluation plots and per-class metrics